In [2]:
from dotenv import load_dotenv
import os

load_dotenv()

os.environ["GEMINI_API_KEY"] = os.getenv("GEMINI_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_API_KEY"] = os.getenv("ARTICULOS_LANGSMITH")
os.environ["LANGCHAIN_PROJECT"] = "Autores de articulos"
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")
os.environ["GITHUB_TOKEN"] = os.getenv("GITHUB_TOKEN")
os.environ["GITHUB_PERSONAL_ACCESS_TOKEN"] = os.getenv("GITHUB_PERSONAL_ACCESS_TOKEN")
os.environ["NVIDIA_API_KEY"] = os.getenv("NVIDIA_API_KEY")


from langchain_openai import ChatOpenAI
from typing import Annotated, TypedDict
from langchain_mcp_adapters.tools import load_mcp_tools
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from langchain_groq import ChatGroq
from langgraph.graph.message import add_messages
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver
from contextlib import AsyncExitStack
from langchain_core.tracers.context import tracing_v2_enabled
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_google_genai import ChatGoogleGenerativeAI
from contextlib import AsyncExitStack

/home/santi/Documentos/LangGraph/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### State

In [3]:
class State(TypedDict):
    messages: Annotated[list, add_messages]

### MCP Server

In [ ]:
import os

# This avoids system's "ammnestia"
system_env = dict(os.environ)

# Merge system environment with GitHub token for the GitHub server
github_env = {**system_env, "GITHUB_PERSONAL_ACCESS_TOKEN": os.getenv("GITHUB_PERSONAL_ACCESS_TOKEN")}

# MCP Client Configuration
client = MultiServerMCPClient(
    {
        "filesystem": {
            "command": "npx",
            "args": ["-y", "@modelcontextprotocol/server-filesystem", "/home/santi/Documentos/LangGraph/"],
            "transport": "stdio",
            "env": system_env # Optional
        },
        "github": {
            "command": "npx",
            "args": ["-y", "@modelcontextprotocol/server-github"],
            "env": github_env, # Merged environment with GitHub token
            "transport": "stdio",
        }
    }
)

In [ ]:
async def run_agent():
    # List to hold all tools from both servers
    all_tools = []
    server_names = ["filesystem", "github"]

    # AsyncExitStack: Dinamic context manager to handle multiple servers and tools
    # Allows us to open and close multiple MCP sessions safely, ensuring proper cleanup even if errors occur.
    async with AsyncExitStack() as stack:

        # Start up servers and load tools
        for server_name in server_names:
            await stack.enter_async_context(client.session(server_name))
            tools = await client.get_tools(server_name=server_name)
            all_tools.extend(tools)

        # Tavily goes here as an external API tool, not MCP, so we instantiate it separately and add to the tools list
        tavily_tool = TavilySearchResults(max_results=3) 
        all_tools.append(tavily_tool)

        # NVIDIA API
        model = ChatOpenAI(
            model="z-ai/glm-5.1",
            api_key=os.getenv("NVIDIA_API_KEY"),
            base_url="https://integrate.api.nvidia.com/v1", # NVIDIA's API URL
            temperature=0.0,
        ).bind_tools(all_tools)

        async def call_model(state: State):

            sys_msg = (
                "system", 
                """Eres un asistente técnico especializado en la navegación de sistemas de archivos y repositorios de GitHub.
                    Tu tarea es ayudar al usuario a buscar en la web y en su sistema de archivos local 
                    para responder a sus preguntas. Utiliza las herramientas disponibles para buscar en la web (tavily), 
                    en el sistema de archivos (filesystem) y en GitHub (github), y proporciona respuestas claras y
                    concisas basadas en la información que encuentres.
                    REGLA CRÍTICA DE ESPACIO: NUNCA utilices la herramienta 'directory_tree' en carpetas raíz
                    o proyectos de código, ya que los entornos virtuales (.venv) y repositorios (.git)
                    saturarán tu memoria. Utiliza siempre 'list_directory' (que es superficial) 
                    para explorar nivel por nivel, y entra solo a las carpetas que te interesen"""
            )

            prompt = [sys_msg] + state["messages"]
            response = await model.ainvoke(prompt)

            return {"messages": [response]}

        # Graph construction        
        graph = StateGraph(State)

        # Nodes
        graph.add_node("agent", call_model)
        graph.add_node("tools", ToolNode(all_tools))

        def should_continue(state: State):
            if state["messages"][-1].tool_calls:
                return "tools"
            return END
        
        # Transitions
        graph.add_edge(START, "agent")
        graph.add_conditional_edges("agent", should_continue)
        graph.add_edge("tools", "agent")

        # Checkpoint de memoria para guardar el estado del grafo en cada paso
        memory = MemorySaver()
        config = {"configurable": {"thread_id": "1"}}

        app = graph.compile(
            checkpointer=memory,
        )
        
        inputs = {"messages": [("user", "Busca en internet el README del repositorio de LangGraph. Luego crea un archivo con esa información y súbelo a GitHub usando las herramientas disponibles. Ten en cuenta que el respositorio al que debes subir el archivo repositorio se encuentra en /home/santi/Documentos/LangGraph/")]}
        result = await app.ainvoke(inputs, config=config)

        for msg in result["messages"]:
            msg.pretty_print()

        return app

app = await run_agent()

/tmp/ipykernel_10554/172589430.py:16: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  tavily_tool = TavilySearchResults(max_results=3)


================================ Human Message =================================

Busca en internet el README del repositorio de LangGraph. Luego crea un archivo con esa información y súbelo a GitHub usando las herramientas disponibles. Ten en cuenta que el respositorio al que debes subir el archivo repositorio se encuentra en /home/santi/Documentos/LangGraph/
================================== Ai Message ==================================

Voy a buscar el README del repositorio de LangGraph en internet y luego lo subiré a tu repositorio local. Empecemos:
Tool Calls:
  tavily_search_results_json (chatcmpl-tool-ae10329189fc1488)
 Call ID: chatcmpl-tool-ae10329189fc1488
  Args:
    query: LangGraph GitHub repository README langchain-ai
  list_directory (chatcmpl-tool-aae31c75e41ad60b)
 Call ID: chatcmpl-tool-aae31c75e41ad60b
  Args:
    path: /home/santi/Documentos/LangGraph
================================= Tool Message =================================
Name: tavily_search_results_json


In [ ]:
from IPython.display import Image, display
display(Image(app.get_graph().draw_mermaid_png()))